# FLUX LoRA Preparation

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**⚠️ Do not run all cells: rename and resize cells permanently modify your images**

## Install

In [1]:
# Install dependencies
!pip install datasets datasets[vision] --quiet
!pip install transformers qwen-vl-utils -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 68.2 MB/s eta 0:00:00


In [ ]:
# Reset (Crash) Environment, then continue to next cell
import importlib, sys, os
os.kill(os.getpid(), 9)

## Setup

In [1]:
# Setup
import os
from PIL import Image
import pandas as pd
from datasets import Dataset

## Mount Drive

In [2]:
# Mount your Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Set Folder

In [3]:
# Set folder
DATASET_DIR = '/content/drive/MyDrive/cartoonify/02_FLUX.1/dataset_FLUX.1'

In [5]:
# Check your current dataset filenames
!ls {DATASET_DIR}

CTN0011.jpg  CTN0014.jpg  CTN0017.jpg  CTN003.jpg  CTN006.jpg  CTN009.jpg
CTN0012.jpg  CTN0015.jpg  CTN001.jpg   CTN004.jpg  CTN007.jpg
CTN0013.jpg  CTN0016.jpg  CTN002.jpg   CTN005.jpg  CTN008.jpg


## Batch Edit Image Sizes
**⚠️ Run cells below with caution — changes to files are irreversible**

In [8]:
# *** Running this cell will resize all images in folder to 1024 x 1024 ***
FILE_TYPE =".jpg"
for f in sorted(os.listdir(DATASET_DIR)):
    if f.endswith(FILE_TYPE):
        path = os.path.join(DATASET_DIR, f)
        img = Image.open(path).resize((1024, 1024), Image.LANCZOS)
        img.save(path)
        print(f"{f} → 1024x1024")

CTN001.jpg → 1024x1024
CTN0011.jpg → 1024x1024
CTN0012.jpg → 1024x1024
CTN0013.jpg → 1024x1024
CTN0014.jpg → 1024x1024
CTN0015.jpg → 1024x1024
CTN0016.jpg → 1024x1024
CTN0017.jpg → 1024x1024
CTN002.jpg → 1024x1024
CTN003.jpg → 1024x1024
CTN004.jpg → 1024x1024
CTN005.jpg → 1024x1024
CTN006.jpg → 1024x1024
CTN007.jpg → 1024x1024
CTN008.jpg → 1024x1024
CTN009.jpg → 1024x1024


In [ ]:
# *** Running this cell will resize all images so the longest edge is 1024 ***
for f in sorted(os.listdir(DATASET_DIR)):
    if f.endswith(FILE_TYPE):
        path = os.path.join(DATASET_DIR, f)
        img = Image.open(path)
        ratio = 1024 / max(img.size)
        new_size = (int(img.width * ratio), int(img.height * ratio))
        img = img.resize(new_size, Image.LANCZOS)
        img.save(path)
        print(f"{f} → {new_size[0]}x{new_size[1]}")

## Create captions as separate text files for FLUX LoRA Training

In [13]:
import gspread
from google.colab import auth
from google.auth import default
import os

# Auth
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open sheet
sh = gc.open('Cartoonify_FLUX_Captions')
ws = sh.worksheet('metadata')

# Get all values
data = ws.get_all_values()

# B1 is header, captions start at B2, filenames in A2+
output_dir = DATASET_DIR
os.makedirs(output_dir, exist_ok=True)

count = 0
for row in data[1:]:  # skip header row
    if len(row) < 2:
        continue
    filename = row[0].strip()  # e.g. bof_aar_lacha_01.png
    caption = row[1].strip()
    if not filename or not caption:
        continue

    # Strip extension and add .txt
    base = os.path.splitext(filename)[0]  # bof_aar_lacha_01
    txt_path = os.path.join(output_dir, base + '.txt')

    with open(txt_path, 'w') as f:
        f.write(caption)
    count += 1

print(f'Saved {count} caption files to {output_dir}')

Saved 16 caption files to /content/drive/MyDrive/cartoonify/02_FLUX.1/dataset_FLUX.1


## Disconnect

In [14]:
from google.colab import runtime
runtime.unassign()